# Final Modeling Pipeline - V6 + Confirmed High-Tail Correction

This notebook rebuilds the trusted V6 model and writes the confirmed final upload file.

- Base model: hierarchical target encoding plus stacked/blended tabular models.
- Validation proxy: `source_id` GroupKFold for ensemble selection.
- Confirmed final postprocess: shrink predictions above 85 by 31.6%.
- Best confirmed public score from this file family: 11.89198.
- Final upload file: `submission.csv`.


## 1. Setup & Load Data

In [1]:
import warnings, gc, time
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor
from scipy.optimize import minimize

TARGET = 'property_organic_content'
t_start = time.time()

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')


y = train[TARGET].values
groups = train['source_id'].values
Y_MIN, Y_MAX = float(np.min(y)), float(np.percentile(y, 99.5))


## 2. Feature Engineering

Build numeric features from spectroscopy, chemistry, coordinates, categorical interactions, and missingness. Raw categorical copies with a leading underscore are kept only for fold-safe target encoding.


In [2]:
# 2. Feature engineering 
def make_indep(df_all):
    X = df_all.drop(columns=[TARGET], errors='ignore').copy()
    X = X.drop(columns=['sample_id'], errors='ignore')
    band_a = [c for c in X.columns if c.startswith('spectral_band_A_PC_')]
    band_b = [c for c in X.columns if c.startswith('spectral_band_B_PC_')]
    eps = 1e-6

    X['missing_total'] = X.isna().sum(axis=1)
    X['band_B_available_actual_num'] = X[band_b].notna().any(axis=1).astype(int)
    X['band_B_missing_count'] = X[band_b].isna().sum(axis=1)
    X['coord_available_num'] = X[['latitude', 'longitude']].notna().all(axis=1).astype(int)
    X['chem_missing_count'] = X[['property_acidity_index', 'cation_Ca', 'cation_Mg',
                                  'cation_Na', 'cation_exchange_capacity']].isna().sum(axis=1)

    for p, cols in [('A', band_a), ('B', band_b)]:
        V = X[cols]
        X[f'{p}_mean'] = V.mean(axis=1)
        X[f'{p}_std'] = V.std(axis=1)
        X[f'{p}_min'] = V.min(axis=1)
        X[f'{p}_max'] = V.max(axis=1)
        X[f'{p}_l2'] = np.sqrt((V ** 2).sum(axis=1))
        X[f'{p}_abs_sum'] = V.abs().sum(axis=1)
        X[f'{p}_max_abs'] = V.abs().max(axis=1)
        for c in cols[:3]:
            X[c + '_abs'] = X[c].abs()
            X[c + '_sq'] = X[c] ** 2

    X['particle_total'] = X['property_particle_coarse'] + X['property_particle_fine']
    X['fine_fraction'] = X['property_particle_fine'] / (X['particle_total'] + eps)
    X['fine_to_coarse'] = X['property_particle_fine'] / (X['property_particle_coarse'] + eps)
    X['fine_minus_coarse'] = X['property_particle_fine'] - X['property_particle_coarse']

    X['base_cation_sum'] = X[['cation_Ca', 'cation_Mg', 'cation_Na']].sum(axis=1)
    X['ca_mg_ratio'] = X['cation_Ca'] / (X['cation_Mg'] + eps)
    X['mg_ca_ratio'] = X['cation_Mg'] / (X['cation_Ca'] + eps)
    X['base_saturation_proxy'] = X['base_cation_sum'] / (X['cation_exchange_capacity'] + eps)
    X['ca_cec_ratio'] = X['cation_Ca'] / (X['cation_exchange_capacity'] + eps)
    X['mg_cec_ratio'] = X['cation_Mg'] / (X['cation_exchange_capacity'] + eps)
    X['cec_per_fine'] = X['cation_exchange_capacity'] / (X['property_particle_fine'] + eps)
    X['acidity_x_cec'] = X['property_acidity_index'] * X['cation_exchange_capacity']
    X['acidity_per_cec'] = X['property_acidity_index'] / (X['cation_exchange_capacity'] + eps)
    for c in ['cation_exchange_capacity', 'cation_Ca', 'cation_Mg', 'property_acidity_index']:
        X['log1p_' + c] = np.log1p(X[c])

    X['abs_latitude'] = X['latitude'].abs()
    X['abs_longitude'] = X['longitude'].abs()
    X['lat_lon_sum'] = X['latitude'] + X['longitude']
    X['lat_lon_diff'] = X['latitude'] - X['longitude']
    X['lat_lon_prod'] = X['latitude'] * X['longitude']
    X['lat_round_1'] = X['latitude'].round(1)
    X['lon_round_1'] = X['longitude'].round(1)

    def inter(cols, name):
        X[name] = X[cols].astype('string').fillna('NA').agg('|'.join, axis=1)

    for cols, name in [
        (['geo_zone_macro', 'geo_zone_meso', 'geo_zone_micro'], 'geo_hierarchy'),
        (['biome', 'land_cover_type'], 'biome_landcover'),
        (['source_id', 'geo_zone_micro'], 'source_micro'),
        (['source_id', 'land_cover_type'], 'source_landcover'),
        (['geo_zone_meso', 'land_cover_type'], 'meso_landcover'),
        (['land_cover_type', 'parent_rock_type'], 'landcover_rock'),
        (['source_id', 'parent_rock_type'], 'source_rock'),
    ]:
        inter(cols, name)
    X['latlon_grid1'] = X['lat_round_1'].astype('string').fillna('NA') + '|' + X['lon_round_1'].astype('string').fillna('NA')

    # simpan kolom kategorikal mentah (sblm encoding) -- dipakai utk anchored TE source_id
    X['_source_id_raw'] = X['source_id'].astype('string')
    X['_geo_zone_macro_raw'] = X['geo_zone_macro'].astype('string')
    X['_biome_raw'] = X['biome'].astype('string')
    X['_parent_rock_type_raw'] = X['parent_rock_type'].astype('string')

    cat_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    cat_cols = [c for c in cat_cols if not c.startswith('_')]
    X_enc = X.copy()
    for c in cat_cols:
        s = X[c].astype('string').fillna('NA')
        freq = s.value_counts(dropna=False)
        X_enc[c + '_freq'] = s.map(freq).astype(float)
        vals = {v: i for i, v in enumerate(s.unique())}
        X_enc[c] = s.map(vals).astype('int32')

    nun = X_enc.nunique(dropna=False)
    drop_const = [c for c in nun[nun <= 1].index.tolist() if not c.startswith('_')]
    X_enc = X_enc.drop(columns=drop_const)
    X_enc = X_enc.replace([np.inf, -np.inf], np.nan)
    return X_enc, cat_cols


all_df = pd.concat([train, test], ignore_index=True, sort=False)
X_all, cat_str_cols = make_indep(all_df)
X = X_all.iloc[:len(train)].reset_index(drop=True)
X_test = X_all.iloc[len(train):].reset_index(drop=True)
RAW_COLS = [c for c in X.columns if c.startswith('_')]

## 3. Fold-Safe Target Encoding

- `add_te_fold`: standard target encoding for categorical columns other than `source_id`, smoothed toward the fold training mean.
- `add_anchored_te_source`: target encoding for `source_id`, smoothed toward a regional prior from geography, biome, and parent rock type.
- `add_te_full`: applies both encoders inside each fold to avoid validation/test leakage.


In [3]:
# 3. Fold-safe target encoding: standard categorical TE plus anchored source_id TE
te_source_cols_full = [
    'source_id', 'geo_zone_macro', 'geo_zone_meso', 'geo_zone_micro',
    'land_cover_type', 'biome', 'parent_rock_type', 'sampling_strategy',
    'geo_hierarchy', 'biome_landcover', 'source_micro', 'source_landcover',
    'meso_landcover', 'landcover_rock', 'source_rock', 'latlon_grid1',
]
HIGH_RISK_TE_COLS = {'source_micro', 'source_landcover', 'source_rock', 'latlon_grid1'}
te_source_cols = [c for c in te_source_cols_full
                   if c in X.columns and c not in HIGH_RISK_TE_COLS]
# source_id is handled separately by the anchored encoder.
te_source_cols_standard = [c for c in te_source_cols if c != 'source_id']


def simple_te_series(cat_tr, y_tr, cat_apply, m, gmean):
    stats = pd.DataFrame({'cat': cat_tr, 'y': y_tr}).groupby('cat')['y'].agg(['count', 'mean'])
    enc = (stats['mean'] * stats['count'] + gmean * m) / (stats['count'] + m)
    return pd.Series(cat_apply).map(enc).fillna(gmean).values, stats


def add_te_fold(X_tr_base, y_tr, X_va_base, X_te_base, cols, ms=(50, 200)):
    """Standard target encoding smoothed toward the fold training mean."""
    X_tr = X_tr_base.copy(); X_va = X_va_base.copy(); X_te = X_te_base.copy()
    gmean = float(np.mean(y_tr))
    for c in cols:
        cat = X_tr_base[c].values
        for m in ms:
            enc_tr, stats = simple_te_series(cat, y_tr, cat, m, gmean)
            X_tr[f'{c}_te_m{m}'] = enc_tr
            X_va[f'{c}_te_m{m}'], _ = simple_te_series(cat, y_tr, X_va_base[c].values, m, gmean)
            X_te[f'{c}_te_m{m}'], _ = simple_te_series(cat, y_tr, X_te_base[c].values, m, gmean)
        cnt = pd.DataFrame({'cat': cat, 'y': y_tr}).groupby('cat')['y'].agg('count')
        X_tr[f'{c}_te_count'] = pd.Series(cat).map(cnt).fillna(0).values
        X_va[f'{c}_te_count'] = pd.Series(X_va_base[c].values).map(cnt).fillna(0).values
        X_te[f'{c}_te_count'] = pd.Series(X_te_base[c].values).map(cnt).fillna(0).values
    return X_tr, X_va, X_te


def add_anchored_te_source(X_tr, X_va, X_te, X_tr_raw, X_va_raw, X_te_raw, y_tr, ms=(50, 200)):
    """Target encode source_id with shrinkage toward a regional prior instead of the global mean."""
    gmean = float(np.mean(y_tr))
    anchor_cols_raw = ['_geo_zone_macro_raw', '_biome_raw', '_parent_rock_type_raw']

    def region_prior_for(df_raw):
        parts = []
        for c in anchor_cols_raw:
            enc, _ = simple_te_series(X_tr_raw[c].values, y_tr, df_raw[c].values, 100, gmean)
            parts.append(enc)
        return np.mean(parts, axis=0)

    region_prior_tr = region_prior_for(X_tr_raw)
    region_prior_va = region_prior_for(X_va_raw)
    region_prior_te = region_prior_for(X_te_raw)

    src_tr = X_tr_raw['_source_id_raw'].values
    stats = pd.DataFrame({'cat': src_tr, 'y': y_tr, 'prior': region_prior_tr}).groupby('cat').agg(
        count=('y', 'count'), mean=('y', 'mean'), prior=('prior', 'mean'))

    for m in ms:
        enc_map = (stats['mean'] * stats['count'] + stats['prior'] * m) / (stats['count'] + m)
        X_tr[f'source_id_anchte_m{m}'] = pd.Series(src_tr).map(enc_map).fillna(gmean).values
        X_va[f'source_id_anchte_m{m}'] = pd.Series(X_va_raw['_source_id_raw'].values).map(enc_map).fillna(gmean).values
        X_te[f'source_id_anchte_m{m}'] = pd.Series(X_te_raw['_source_id_raw'].values).map(enc_map).fillna(gmean).values
    cnt = stats['count']
    X_tr['source_id_te_count'] = pd.Series(src_tr).map(cnt).fillna(0).values
    X_va['source_id_te_count'] = pd.Series(X_va_raw['_source_id_raw'].values).map(cnt).fillna(0).values
    X_te['source_id_te_count'] = pd.Series(X_te_raw['_source_id_raw'].values).map(cnt).fillna(0).values
    # Keep the regional prior as an additional feature.
    X_tr['region_prior'] = region_prior_tr
    X_va['region_prior'] = region_prior_va
    X_te['region_prior'] = region_prior_te
    return X_tr, X_va, X_te


def add_te_full(X_tr0, y_tr, X_va0, X_te0):
    """Apply standard TE for non-source columns and anchored TE for source_id."""
    X_tr, X_va, X_te = add_te_fold(X_tr0, y_tr, X_va0, X_te0, te_source_cols_standard)
    X_tr, X_va, X_te = add_anchored_te_source(
        X_tr, X_va, X_te, X_tr0, X_va0, X_te0, y_tr)
    return X_tr, X_va, X_te


def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


results = []; oofs = {}; preds = {}

## 3b. Target-Encoding Sanity Check

Compare standard `source_id` target encoding against anchored `source_id` target encoding using `source_id` GroupKFold. A negative delta means the anchored encoder is better.


In [4]:
# 3b. Sanity check: compare standard and anchored source_id TE with GroupKFold.
import lightgbm as lgb

def quick_probe():
    bins = pd.qcut(y, 10, labels=False, duplicates='drop')
    params = dict(n_estimators=2500, num_leaves=31, learning_rate=0.03, min_child_samples=20,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1)

    cv = GroupKFold(n_splits=5)
    oof_std = np.zeros(len(y)); oof_anch = np.zeros(len(y))
    for tr_idx, va_idx in cv.split(X, y, groups):
        X_tr0 = X.iloc[tr_idx].reset_index(drop=True); X_va0 = X.iloc[va_idx].reset_index(drop=True)
        y_tr = y[tr_idx]

        # Standard path: source_id is smoothed toward the fold training mean.
        X_tr_s, X_va_s, _ = add_te_fold(X_tr0, y_tr, X_va0, X_test, te_source_cols)
        X_tr_s = X_tr_s.drop(columns=RAW_COLS); X_va_s = X_va_s.drop(columns=RAW_COLS)
        m1 = lgb.LGBMRegressor(**params, random_state=91, n_jobs=4, verbosity=-1)
        m1.fit(X_tr_s, y_tr, eval_set=[(X_va_s, y[va_idx])],
               callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        oof_std[va_idx] = np.clip(m1.predict(X_va_s), 0, None)

        # Anchored path.
        X_tr_a, X_va_a, _ = add_te_full(X_tr0, y_tr, X_va0, X_test)
        X_tr_a = X_tr_a.drop(columns=RAW_COLS); X_va_a = X_va_a.drop(columns=RAW_COLS)
        m2 = lgb.LGBMRegressor(**params, random_state=91, n_jobs=4, verbosity=-1)
        m2.fit(X_tr_a, y_tr, eval_set=[(X_va_a, y[va_idx])],
               callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        oof_anch[va_idx] = np.clip(m2.predict(X_va_a), 0, None)

    print('[SANITY CHECK] GroupKFold RMSE, TE source_id STANDAR (shrink ke global) :', rmse(y, oof_std))
    print('[SANITY CHECK] GroupKFold RMSE, TE source_id ANCHORED (shrink ke region):', rmse(y, oof_anch))
    print('[SANITY CHECK] Selisih (anchored - standar), negatif = anchored lebih baik:', rmse(y, oof_anch) - rmse(y, oof_std))

quick_probe()
print(f'[checkpoint] sanity check selesai, elapsed {time.time()-t_start:.0f}s', flush=True)

[SANITY CHECK] GroupKFold RMSE, TE source_id STANDAR (shrink ke global) : 18.458241567219495
[SANITY CHECK] GroupKFold RMSE, TE source_id ANCHORED (shrink ke region): 18.75318227503204
[SANITY CHECK] Selisih (anchored - standar), negatif = anchored lebih baik: 0.2949407078125432
[checkpoint] sanity check selesai, elapsed 108s


## 4. CatBoost Models


In [5]:
# ---------------------------------------------------------------------------
# 4. CatBoost models
# ---------------------------------------------------------------------------
def run_cat(name, params, seeds=(91, 202), n_splits=5, log_target=False):
    bins = pd.qcut(y, 10, labels=False, duplicates='drop')
    oof_accum = np.zeros(len(y))
    pred_accum = np.zeros(len(test))
    rows = []
    y_use_full = np.log1p(y) if log_target else y
    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        oof = np.zeros(len(y)); pred = np.zeros(len(test))
        for fold, (tr_idx, va_idx) in enumerate(cv.split(X, bins), 1):
            X_tr0 = X.iloc[tr_idx].reset_index(drop=True)
            X_va0 = X.iloc[va_idx].reset_index(drop=True)
            X_te0 = X_test.copy()
            y_tr = y_use_full[tr_idx]; y_va_eval = y[va_idx]

            X_tr, X_va, X_te = add_te_full(X_tr0, y_tr, X_va0, X_te0)
            X_tr = X_tr.drop(columns=RAW_COLS); X_va = X_va.drop(columns=RAW_COLS); X_te = X_te.drop(columns=RAW_COLS)

            model = CatBoostRegressor(**params, random_seed=seed + fold,
                                       allow_writing_files=False, verbose=False, thread_count=4)
            model.fit(X_tr, y_tr, eval_set=(X_va, y_use_full[va_idx]), early_stopping_rounds=200, verbose=False)

            pv_raw = model.predict(X_va); pt_raw = model.predict(X_te)
            if log_target:
                pv = np.clip(np.expm1(pv_raw), 0, None); pt = np.clip(np.expm1(pt_raw), 0, None)
            else:
                pv = np.clip(pv_raw, 0, None); pt = np.clip(pt_raw, 0, None)
            oof[va_idx] = pv
            pred += pt / n_splits
            row = {'fold': fold, 'seed': seed, 'rmse': rmse(y_va_eval, pv),
                   'best_iter': model.get_best_iteration()}
            rows.append(row)
            print(name, row, flush=True)
            del model; gc.collect()
        oof_accum += oof / len(seeds)
        pred_accum += pred / len(seeds)
    sc = {'name': name, 'rmse': rmse(y, oof_accum), 'mae': mean_absolute_error(y, oof_accum),
          'r2': r2_score(y, oof_accum), 'rows': rows}
    print('DONE', name, sc['rmse'], sc['mae'], sc['r2'], flush=True)
    return (sc, oof_accum, pred_accum)


params_d6 = dict(iterations=2500, depth=6, learning_rate=0.035, l2_leaf_reg=10,
                  random_strength=1.5, bagging_temperature=0.5, bootstrap_type='Bayesian',
                  loss_function='RMSE', eval_metric='RMSE')
params_d7_deep = dict(iterations=2500, depth=7, learning_rate=0.03, l2_leaf_reg=8,
                       random_strength=1.0, bagging_temperature=0.3, bootstrap_type='Bayesian',
                       loss_function='RMSE', eval_metric='RMSE')

for name, params, seeds, log_t in [
    ('cat_num_d6', params_d6, (91, 202), False),
    ('cat_num_d7_deep', params_d7_deep, (91, 202), False),
]:
    m, o, p = run_cat(name, params, seeds=seeds, log_target=log_t)
    results.append(m); oofs[name] = o; preds[name] = p
    np.savez('data/checkpoint_v6.npz',
             **{f'oof_{k}': v for k, v in oofs.items()},
             **{f'pred_{k}': v for k, v in preds.items()})
    print(f'[checkpoint saved after {name}] elapsed {time.time()-t_start:.0f}s', flush=True)

cat_num_d6 {'fold': 1, 'seed': 91, 'rmse': 11.981588618866166, 'best_iter': 2451}
cat_num_d6 {'fold': 2, 'seed': 91, 'rmse': 10.990977746900057, 'best_iter': 2495}
cat_num_d6 {'fold': 3, 'seed': 91, 'rmse': 11.321556867008699, 'best_iter': 2459}
cat_num_d6 {'fold': 4, 'seed': 91, 'rmse': 10.861170296767188, 'best_iter': 2499}
cat_num_d6 {'fold': 5, 'seed': 91, 'rmse': 11.341748958787498, 'best_iter': 2475}
cat_num_d6 {'fold': 1, 'seed': 202, 'rmse': 10.92523413987892, 'best_iter': 2458}
cat_num_d6 {'fold': 2, 'seed': 202, 'rmse': 11.268352309712984, 'best_iter': 2493}
cat_num_d6 {'fold': 3, 'seed': 202, 'rmse': 11.180392795082692, 'best_iter': 2490}
cat_num_d6 {'fold': 4, 'seed': 202, 'rmse': 11.408423691563453, 'best_iter': 2416}
cat_num_d6 {'fold': 5, 'seed': 202, 'rmse': 11.728137552328153, 'best_iter': 2495}
DONE cat_num_d6 11.220914946896274 6.976481033453289 0.7660239707309069
[checkpoint saved after cat_num_d6] elapsed 1197s
cat_num_d7_deep {'fold': 1, 'seed': 91, 'rmse': 12.004

## 5. Ridge Model


In [6]:
# 5. Ridge model for linear diversity
def run_ridge(name, alpha=40.0, n_splits=5, seed=303):
    bins = pd.qcut(y, 10, labels=False, duplicates='drop')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y)); pred = np.zeros(len(test)); rows = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, bins), 1):
        X_tr0 = X.iloc[tr_idx].reset_index(drop=True)
        X_va0 = X.iloc[va_idx].reset_index(drop=True)
        X_te0 = X_test.copy()
        y_tr = y[tr_idx]; y_va_eval = y[va_idx]

        X_tr, X_va, X_te = add_te_full(X_tr0, y_tr, X_va0, X_te0)
        X_tr = X_tr.drop(columns=RAW_COLS); X_va = X_va.drop(columns=RAW_COLS); X_te = X_te.drop(columns=RAW_COLS)

        med = X_tr.median(numeric_only=True)
        X_tr_f = X_tr.fillna(med); X_va_f = X_va.fillna(med); X_te_f = X_te.fillna(med)

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr_f)
        X_va_s = scaler.transform(X_va_f)
        X_te_s = scaler.transform(X_te_f)

        model = Ridge(alpha=alpha, random_state=seed)
        model.fit(X_tr_s, y_tr)
        pv = np.clip(model.predict(X_va_s), 0, None)
        pt = np.clip(model.predict(X_te_s), 0, None)
        oof[va_idx] = pv; pred += pt / n_splits
        row = {'fold': fold, 'rmse': rmse(y_va_eval, pv)}
        rows.append(row)
        print(name, row, flush=True)
    sc = {'name': name, 'rmse': rmse(y, oof), 'mae': mean_absolute_error(y, oof),
          'r2': r2_score(y, oof), 'rows': rows}
    print('DONE', name, sc['rmse'], sc['mae'], sc['r2'], flush=True)
    return (sc, oof, pred)


m_ridge, o_ridge, p_ridge = run_ridge('ridge_te40', alpha=40.0)
results.append(m_ridge); oofs['ridge_te40'] = o_ridge; preds['ridge_te40'] = p_ridge
np.savez('data/checkpoint_v6.npz',
         **{f'oof_{k}': v for k, v in oofs.items()},
         **{f'pred_{k}': v for k, v in preds.items()})
print(f'[checkpoint saved after ridge] elapsed {time.time()-t_start:.0f}s', flush=True)

ridge_te40 {'fold': 1, 'rmse': 14.734197698563685}
ridge_te40 {'fold': 2, 'rmse': 14.572246192275546}
ridge_te40 {'fold': 3, 'rmse': 14.418601711362896}
ridge_te40 {'fold': 4, 'rmse': 14.835822960016307}
ridge_te40 {'fold': 5, 'rmse': 13.787274515338341}
DONE ridge_te40 14.474346946800818 9.386256506842424 0.6106745608267103
[checkpoint saved after ridge] elapsed 2182s


## 6. LightGBM Models


In [7]:
def run_lgb(name, params, seeds=(91,), n_splits=5, log_target=False):
    bins = pd.qcut(y, 10, labels=False, duplicates='drop')
    oof_accum = np.zeros(len(y)); pred_accum = np.zeros(len(test)); rows = []
    y_use_full = np.log1p(y) if log_target else y
    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        oof = np.zeros(len(y)); pred = np.zeros(len(test))
        for fold, (tr_idx, va_idx) in enumerate(cv.split(X, bins), 1):
            X_tr0 = X.iloc[tr_idx].reset_index(drop=True)
            X_va0 = X.iloc[va_idx].reset_index(drop=True)
            X_te0 = X_test.copy()
            y_tr = y_use_full[tr_idx]; y_va_eval = y[va_idx]

            X_tr, X_va, X_te = add_te_full(X_tr0, y_tr, X_va0, X_te0)
            X_tr = X_tr.drop(columns=RAW_COLS); X_va = X_va.drop(columns=RAW_COLS); X_te = X_te.drop(columns=RAW_COLS)

            model = lgb.LGBMRegressor(**params, random_state=seed + fold, n_jobs=4, verbosity=-1)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_use_full[va_idx])],
                      callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])

            pv_raw = model.predict(X_va); pt_raw = model.predict(X_te)
            if log_target:
                pv = np.clip(np.expm1(pv_raw), 0, None); pt = np.clip(np.expm1(pt_raw), 0, None)
            else:
                pv = np.clip(pv_raw, 0, None); pt = np.clip(pt_raw, 0, None)
            oof[va_idx] = pv; pred += pt / n_splits
            row = {'fold': fold, 'seed': seed, 'rmse': rmse(y_va_eval, pv),
                   'best_iter': model.best_iteration_}
            rows.append(row)
            print(name, row, flush=True)
            del model; gc.collect()
        oof_accum += oof / len(seeds)
        pred_accum += pred / len(seeds)
    sc = {'name': name, 'rmse': rmse(y, oof_accum), 'mae': mean_absolute_error(y, oof_accum),
          'r2': r2_score(y, oof_accum), 'rows': rows}
    print('DONE', name, sc['rmse'], sc['mae'], sc['r2'], flush=True)
    return (sc, oof_accum, pred_accum)


lgb_params_base = dict(n_estimators=3000, num_leaves=31, max_depth=-1, learning_rate=0.03,
                        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                        reg_alpha=0.1, reg_lambda=0.1, objective='regression')
lgb_params_extra = dict(n_estimators=3000, num_leaves=255, max_depth=-1, learning_rate=0.015,
                         min_child_samples=15, subsample=0.85, colsample_bytree=0.6,
                         reg_alpha=0.05, reg_lambda=0.05, objective='regression')

for name, params, seeds, log_t in [
    ('lgb_base', lgb_params_base, (91, 404), False),
    ('lgb_base_log', lgb_params_base, (91, 404), True),
    ('lgb_extra_wide', lgb_params_extra, (91, 404), False),
]:
    m, o, p = run_lgb(name, params, seeds=seeds, log_target=log_t)
    results.append(m); oofs[name] = o; preds[name] = p
    np.savez('data/checkpoint_v6.npz',
             **{f'oof_{k}': v for k, v in oofs.items()},
             **{f'pred_{k}': v for k, v in preds.items()})
    print(f'[checkpoint saved after {name}] elapsed {time.time()-t_start:.0f}s', flush=True)

lgb_base {'fold': 1, 'seed': 91, 'rmse': 12.083172372062657, 'best_iter': 872}
lgb_base {'fold': 2, 'seed': 91, 'rmse': 10.824374730276043, 'best_iter': 1514}
lgb_base {'fold': 3, 'seed': 91, 'rmse': 11.510608256232459, 'best_iter': 818}
lgb_base {'fold': 4, 'seed': 91, 'rmse': 10.735724670483773, 'best_iter': 1425}
lgb_base {'fold': 5, 'seed': 91, 'rmse': 11.458127554456064, 'best_iter': 1230}
lgb_base {'fold': 1, 'seed': 404, 'rmse': 11.288851429786751, 'best_iter': 1273}
lgb_base {'fold': 2, 'seed': 404, 'rmse': 11.97821994915764, 'best_iter': 844}
lgb_base {'fold': 3, 'seed': 404, 'rmse': 11.753665981955624, 'best_iter': 1251}
lgb_base {'fold': 4, 'seed': 404, 'rmse': 10.437970623381796, 'best_iter': 750}
lgb_base {'fold': 5, 'seed': 404, 'rmse': 11.295638341257884, 'best_iter': 1347}
DONE lgb_base 11.207147370677607 6.963439562993763 0.7665977753884329
[checkpoint saved after lgb_base] elapsed 2236s
lgb_base_log {'fold': 1, 'seed': 91, 'rmse': 12.441719532492316, 'best_iter': 1690

## 7. XGBoost

In [8]:
import xgboost as xgb

def run_xgb(name, params, seeds=(91,), n_splits=5):
    bins = pd.qcut(y, 10, labels=False, duplicates='drop')
    oof_accum = np.zeros(len(y)); pred_accum = np.zeros(len(test)); rows = []
    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        oof = np.zeros(len(y)); pred = np.zeros(len(test))
        for fold, (tr_idx, va_idx) in enumerate(cv.split(X, bins), 1):
            X_tr0 = X.iloc[tr_idx].reset_index(drop=True)
            X_va0 = X.iloc[va_idx].reset_index(drop=True)
            X_te0 = X_test.copy()
            y_tr = y[tr_idx]; y_va = y[va_idx]

            X_tr, X_va, X_te = add_te_full(X_tr0, y_tr, X_va0, X_te0)
            X_tr = X_tr.drop(columns=RAW_COLS); X_va = X_va.drop(columns=RAW_COLS); X_te = X_te.drop(columns=RAW_COLS)

            model = xgb.XGBRegressor(**params, random_state=seed + fold, n_jobs=4,
                                      early_stopping_rounds=200, eval_metric='rmse')
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            pv = np.clip(model.predict(X_va), 0, None)
            pt = np.clip(model.predict(X_te), 0, None)
            oof[va_idx] = pv; pred += pt / n_splits
            row = {'fold': fold, 'seed': seed, 'rmse': rmse(y_va, pv),
                   'best_iter': model.best_iteration}
            rows.append(row)
            print(name, row, flush=True)
            del model; gc.collect()
        oof_accum += oof / len(seeds)
        pred_accum += pred / len(seeds)
    sc = {'name': name, 'rmse': rmse(y, oof_accum), 'mae': mean_absolute_error(y, oof_accum),
          'r2': r2_score(y, oof_accum), 'rows': rows}
    print('DONE', name, sc['rmse'], sc['mae'], sc['r2'], flush=True)
    return (sc, oof_accum, pred_accum)


xgb_params_d4 = dict(n_estimators=3000, max_depth=4, learning_rate=0.03,
                      subsample=0.8, colsample_bytree=0.8,
                      reg_alpha=0.1, reg_lambda=1.0, objective='reg:squarederror')

m_xgb, o_xgb, p_xgb = run_xgb('xgb_d4', xgb_params_d4, seeds=(91, 404))
results.append(m_xgb); oofs['xgb_d4'] = o_xgb; preds['xgb_d4'] = p_xgb
np.savez('data/checkpoint_v6.npz',
         **{f'oof_{k}': v for k, v in oofs.items()},
         **{f'pred_{k}': v for k, v in preds.items()})
print(f'[checkpoint saved after xgb] elapsed {time.time()-t_start:.0f}s', flush=True)

xgb_d4 {'fold': 1, 'seed': 91, 'rmse': 11.80163598168983, 'best_iter': 2830}
xgb_d4 {'fold': 2, 'seed': 91, 'rmse': 11.053233330813942, 'best_iter': 1968}
xgb_d4 {'fold': 3, 'seed': 91, 'rmse': 11.566261172946977, 'best_iter': 2950}
xgb_d4 {'fold': 4, 'seed': 91, 'rmse': 10.973243570836726, 'best_iter': 2767}
xgb_d4 {'fold': 5, 'seed': 91, 'rmse': 11.37139277407236, 'best_iter': 1767}
xgb_d4 {'fold': 1, 'seed': 404, 'rmse': 11.347190576113032, 'best_iter': 1798}
xgb_d4 {'fold': 2, 'seed': 404, 'rmse': 12.111183273384768, 'best_iter': 1409}
xgb_d4 {'fold': 3, 'seed': 404, 'rmse': 11.785163612157064, 'best_iter': 1162}
xgb_d4 {'fold': 4, 'seed': 404, 'rmse': 10.82771280419484, 'best_iter': 1759}
xgb_d4 {'fold': 5, 'seed': 404, 'rmse': 11.145104339655385, 'best_iter': 1815}
DONE xgb_d4 11.277175028287177 7.033175931565629 0.7636718433860379
[checkpoint saved after xgb] elapsed 2679s


## 8. HistGradientBoostingRegressor Model


In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor

def run_hgb(name, n_splits=5, seed=91):
    bins = pd.qcut(y, 10, labels=False, duplicates='drop')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y)); pred = np.zeros(len(test)); rows = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, bins), 1):
        X_tr0 = X.iloc[tr_idx].reset_index(drop=True)
        X_va0 = X.iloc[va_idx].reset_index(drop=True)
        X_te0 = X_test.copy()
        y_tr = y[tr_idx]; y_va = y[va_idx]

        X_tr, X_va, X_te = add_te_full(X_tr0, y_tr, X_va0, X_te0)
        X_tr = X_tr.drop(columns=RAW_COLS); X_va = X_va.drop(columns=RAW_COLS); X_te = X_te.drop(columns=RAW_COLS)

        model = HistGradientBoostingRegressor(
            max_iter=1500, max_depth=6, learning_rate=0.04, l2_regularization=0.5,
            min_samples_leaf=25, early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=100, random_state=seed + fold)
        model.fit(X_tr, y_tr)

        pv = np.clip(model.predict(X_va), 0, None)
        pt = np.clip(model.predict(X_te), 0, None)
        oof[va_idx] = pv; pred += pt / n_splits
        row = {'fold': fold, 'rmse': rmse(y_va, pv)}
        rows.append(row)
        print(name, row, flush=True)
    sc = {'name': name, 'rmse': rmse(y, oof), 'mae': mean_absolute_error(y, oof),
          'r2': r2_score(y, oof), 'rows': rows}
    print('DONE', name, sc['rmse'], sc['mae'], sc['r2'], flush=True)
    return (sc, oof, pred)


m_hgb, o_hgb, p_hgb = run_hgb('hgb')
results.append(m_hgb); oofs['hgb'] = o_hgb; preds['hgb'] = p_hgb
np.savez('data/checkpoint_v6.npz',
         **{f'oof_{k}': v for k, v in oofs.items()},
         **{f'pred_{k}': v for k, v in preds.items()})
print(f'[checkpoint saved after hgb] elapsed {time.time()-t_start:.0f}s', flush=True)

print()
print(pd.DataFrame([{k: v for k, v in r.items() if k != 'rows'} for r in results]).sort_values('rmse'))

hgb {'fold': 1, 'rmse': 12.385627033220894}
hgb {'fold': 2, 'rmse': 11.371834015764799}
hgb {'fold': 3, 'rmse': 11.58197273026949}
hgb {'fold': 4, 'rmse': 11.078115830169278}
hgb {'fold': 5, 'rmse': 11.409303555473944}
DONE hgb 11.573774804967144 7.240595505237943 0.7510770832169116
[checkpoint saved after hgb] elapsed 2698s

              name       rmse       mae        r2
1  cat_num_d7_deep  11.200486  6.921023  0.766875
3         lgb_base  11.207147  6.963440  0.766598
0       cat_num_d6  11.220915  6.976481  0.766024
5   lgb_extra_wide  11.271812  6.951975  0.763897
6           xgb_d4  11.277175  7.033176  0.763672
4     lgb_base_log  11.466809  6.885937  0.755657
7              hgb  11.573775  7.240596  0.751077
2       ridge_te40  14.474347  9.386257  0.610675


## 9. Ensemble Selection

Four ensemble strategies are evaluated with `source_id` GroupKFold:
1. `linear_standard`: constrained linear blend fit on pooled OOF predictions.
2. `linear_group_aware`: constrained linear blend optimized against grouped folds.
3. `stack_ridge`: Ridge meta-model on OOF predictions plus meta-features.
4. `stack_lgb`: shallow LightGBM meta-model on OOF predictions plus meta-features.


In [10]:
names = list(oofs.keys())
oof_list = [oofs[n] for n in names]
pred_list = [preds[n] for n in names]

O = np.vstack(oof_list).T
P = np.vstack(pred_list).T

# Extra stacking features: source_id frequency and total missing count.
src_count_map = train['source_id'].map(train['source_id'].value_counts())
meta_extra_train = np.column_stack([
    np.log1p(src_count_map.values),
    X['missing_total'].values,
])
src_count_map_test = test['source_id'].map(train['source_id'].value_counts()).fillna(1)
meta_extra_test = np.column_stack([
    np.log1p(src_count_map_test.values),
    X_test['missing_total'].values,
])


def rmse_fn(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


def fit_weights(O_sub, y_sub, l2=1e-3):
    n_models = O_sub.shape[1]
    def obj(w):
        uniform = np.ones(n_models) / n_models
        return rmse_fn(y_sub, O_sub.dot(w)) + l2 * np.sum((w - uniform) ** 2)
    res = minimize(obj, np.ones(n_models) / n_models, bounds=[(0, 1)] * n_models,
                    constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1}, method='SLSQP')
    return res.x


def fit_weights_group_aware(O_full, y_full, groups_full, l2=1e-3, n_splits=5):
    n_models = O_full.shape[1]
    gkf = GroupKFold(n_splits=n_splits)
    splits = list(gkf.split(O_full, y_full, groups_full))
    def obj(w):
        uniform = np.ones(n_models) / n_models
        fold_rmses = [rmse_fn(y_full[va], O_full[va].dot(w)) for _, va in splits]
        return np.mean(fold_rmses) + l2 * np.sum((w - uniform) ** 2)
    res = minimize(obj, np.ones(n_models) / n_models, bounds=[(0, 1)] * n_models,
                    constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1}, method='SLSQP')
    return res.x


def fit_stack_ridge(O_sub, meta_sub, y_sub, alpha=5.0):
    feats = np.column_stack([O_sub, meta_sub])
    scaler = StandardScaler()
    feats_s = scaler.fit_transform(feats)
    model = Ridge(alpha=alpha, positive=False)
    model.fit(feats_s, y_sub)
    return model, scaler


def predict_stack_ridge(model, scaler, O_sub, meta_sub):
    feats = np.column_stack([O_sub, meta_sub])
    feats_s = scaler.transform(feats)
    return np.clip(model.predict(feats_s), 0, None)


def fit_stack_lgb(O_sub, meta_sub, y_sub, seed=42):
    feats = np.column_stack([O_sub, meta_sub])
    model = lgb.LGBMRegressor(n_estimators=300, num_leaves=7, max_depth=3,
                               learning_rate=0.03, min_child_samples=30,
                               reg_alpha=1.0, reg_lambda=2.0, subsample=0.8,
                               colsample_bytree=0.8, random_state=seed, verbosity=-1)
    model.fit(feats, y_sub)
    return model


def predict_stack_lgb(model, O_sub, meta_sub):
    feats = np.column_stack([O_sub, meta_sub])
    return np.clip(model.predict(feats), 0, None)


# --- Evaluasi 4 strategi blend via nested GroupKFold (metrik tervalidasi) ---
gkf_outer = GroupKFold(n_splits=5)
scores_linear_std = []; scores_linear_ga = []; scores_stack_ridge = []; scores_stack_lgb = []

for tr_idx, va_idx in gkf_outer.split(O, y, groups):
    O_tr, O_va = O[tr_idx], O[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    g_tr = groups[tr_idx]
    meta_tr, meta_va = meta_extra_train[tr_idx], meta_extra_train[va_idx]

    w_std = fit_weights(O_tr, y_tr)
    scores_linear_std.append(rmse_fn(y_va, O_va.dot(w_std)))

    w_ga = fit_weights_group_aware(O_tr, y_tr, g_tr)
    scores_linear_ga.append(rmse_fn(y_va, O_va.dot(w_ga)))

    m_ridge_stack, sc_ridge = fit_stack_ridge(O_tr, meta_tr, y_tr)
    pred_ridge = predict_stack_ridge(m_ridge_stack, sc_ridge, O_va, meta_va)
    scores_stack_ridge.append(rmse_fn(y_va, pred_ridge))

    m_lgb_stack = fit_stack_lgb(O_tr, meta_tr, y_tr)
    pred_lgb_stack = predict_stack_lgb(m_lgb_stack, O_va, meta_va)
    scores_stack_lgb.append(rmse_fn(y_va, pred_lgb_stack))

est_linear_std = float(np.mean(scores_linear_std))
est_linear_ga = float(np.mean(scores_linear_ga))
est_stack_ridge = float(np.mean(scores_stack_ridge))
est_stack_lgb = float(np.mean(scores_stack_lgb))

print()
print('Estimasi GroupKFold per strategi blend')
print(f'  Linear blend (standard fit)     : {est_linear_std:.5f}')
print(f'  Linear blend (group-aware fit)  : {est_linear_ga:.5f}')
print(f'  Stacking Ridge + meta-features  : {est_stack_ridge:.5f}')
print(f'  Stacking LightGBM + meta-feats  : {est_stack_lgb:.5f}')

strategies = {
    'linear_standard': est_linear_std,
    'linear_group_aware': est_linear_ga,
    'stack_ridge': est_stack_ridge,
    'stack_lgb': est_stack_lgb,
}
best_strategy = min(strategies, key=strategies.get)
print(f'STRATEGI TERPILIH: {best_strategy} (estimasi {strategies[best_strategy]:.5f})')

# --- Fit strategi terpilih di FULL data utk submission final ---
if best_strategy == 'linear_standard':
    w_final = fit_weights(O, y)
    ens_oof = O.dot(w_final)
    ens_pred = P.dot(w_final)
    print('Bobot final:')
    for n, ww in sorted(zip(names, w_final), key=lambda x: -x[1]):
        print(f'  {n:20s} {ww:.4f}')
elif best_strategy == 'linear_group_aware':
    w_final = fit_weights_group_aware(O, y, groups)
    ens_oof = O.dot(w_final)
    ens_pred = P.dot(w_final)
    print('Bobot final:')
    for n, ww in sorted(zip(names, w_final), key=lambda x: -x[1]):
        print(f'  {n:20s} {ww:.4f}')
elif best_strategy == 'stack_ridge':
    model_final, scaler_final = fit_stack_ridge(O, meta_extra_train, y)
    ens_oof = predict_stack_ridge(model_final, scaler_final, O, meta_extra_train)
    ens_pred = predict_stack_ridge(model_final, scaler_final, P, meta_extra_test)
else:
    model_final = fit_stack_lgb(O, meta_extra_train, y)
    ens_oof = predict_stack_lgb(model_final, O, meta_extra_train)
    ens_pred = predict_stack_lgb(model_final, P, meta_extra_test)


Estimasi GroupKFold per strategi blend
  Linear blend (standard fit)     : 12.03805
  Linear blend (group-aware fit)  : 12.04469
  Stacking Ridge + meta-features  : 11.98828
  Stacking LightGBM + meta-feats  : 12.26387
STRATEGI TERPILIH: stack_ridge (estimasi 11.98828)


## 10. Final Submission

Write the raw V6 base predictions to `data/submission_v6.csv`, then write the confirmed high-tail upload file to `submission.csv`.

Final postprocess: `prediction = prediction - 0.316 * max(prediction - 85, 0)`.


In [11]:
TAIL_THRESHOLD = 85.0
TAIL_ALPHA = 0.316
BASE_SUBMISSION_PATH = 'data/submission_v6.csv'
ROOT_SUBMISSION_PATH = 'submission.csv'


sub = sample.copy()
ens_pred_clipped = np.clip(ens_pred, 0, Y_MAX)
sub[TARGET] = ens_pred_clipped
sub.to_csv(BASE_SUBMISSION_PATH, index=False)

tail_adjusted = ens_pred_clipped - TAIL_ALPHA * np.maximum(ens_pred_clipped - TAIL_THRESHOLD, 0)
tail_adjusted = np.clip(tail_adjusted, 0, None)

final_sub = sample.copy()
final_sub[TARGET] = tail_adjusted
final_sub.to_csv(ROOT_SUBMISSION_PATH, index=False)

assert list(final_sub.columns) == list(sample.columns)
assert len(final_sub) == len(sample)
assert final_sub['sample_id'].equals(sample['sample_id'])
assert np.isfinite(final_sub[TARGET]).all()
assert (final_sub[TARGET] >= 0).all()

summary_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'rows'} for r in results])
summary_df = pd.concat([
    summary_df,
    pd.DataFrame([
        {'name': f'ensemble_v6_{k}', 'rmse': v, 'mae': np.nan, 'r2': np.nan}
        for k, v in strategies.items()
    ]),
    pd.DataFrame([{'name': f'ensemble_v6_CHOSEN_{best_strategy}', 'rmse': strategies[best_strategy],
                    'mae': np.nan, 'r2': np.nan}]),
    pd.DataFrame([{'name': 'final_high_tail_t85_a0p316_public_confirmed', 'rmse': np.nan,
                    'mae': np.nan, 'r2': np.nan}]),
], ignore_index=True)
summary_df.to_csv('data/model_summary_v6.csv', index=False)

print('Finished.')
print(f'GroupKFold estimate, strategy {best_strategy}: {strategies[best_strategy]:.5f}')
print(f'Final postprocess: threshold={TAIL_THRESHOLD:g}, alpha={TAIL_ALPHA:.2f}')
print(f'Wrote base submission: {BASE_SUBMISSION_PATH}')
print(f'Wrote final submission: {ROOT_SUBMISSION_PATH}')
print(f'Total time: {time.time()-t_start:.0f}s')
print(summary_df)


Finished.
GroupKFold estimate, strategy stack_ridge: 11.98828
Final postprocess: threshold=85, alpha=0.32
Wrote base submission: data/submission_v6.csv
Wrote final submission: submission.csv
Total time: 2699s
                                           name       rmse       mae        r2
0                                    cat_num_d6  11.220915  6.976481  0.766024
1                               cat_num_d7_deep  11.200486  6.921023  0.766875
2                                    ridge_te40  14.474347  9.386257  0.610675
3                                      lgb_base  11.207147  6.963440  0.766598
4                                  lgb_base_log  11.466809  6.885937  0.755657
5                                lgb_extra_wide  11.271812  6.951975  0.763897
6                                        xgb_d4  11.277175  7.033176  0.763672
7                                           hgb  11.573775  7.240596  0.751077
8                   ensemble_v6_linear_standard  12.038053       NaN       NaN
9